# Import all needed Modules and set constants (copied from course)

## Sizes
64 was the base in the course. I want to go as low as 16 since at least humans can still recognize patterns in colored images with a size of 4 by 4 (Check out https://de.sporcle.com/games/ohicanchat/name-the-league-of-legends-characters-from-16-pixels if you know League of Legends and want a challenge) Since our AI has to work on grayscale (I think) this is a bit harsh, but maybe 16 by 16 will yield results.

In [28]:
import cv2
import json
from matplotlib import pyplot as plt
import numpy as np
import os
import random

# import a lot of things from keras:
# sequential model
from keras.models import Sequential

# layers
from keras.layers import Input, Dense, Dropout, Flatten, Conv2D, MaxPooling2D, RandomFlip, RandomRotation, RandomContrast, RandomBrightness

# loss function
from keras.metrics import categorical_crossentropy

# callback functions
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

# convert data to categorial vector representation
from keras.utils import to_categorical

# nice progress bar for loading data
from tqdm import tqdm

# helper function for train/test split
from sklearn.model_selection import train_test_split

# import confusion matrix helper function
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# import pre-trained model
from keras.applications.vgg16 import VGG16

# include only those gestures
CONDITIONS = ['like', 'stop']

# image size
# Use a broad array of selections
# Is 16 even viable? We  will find out!
# I really want to do 512 as well, but I will first see how long training takes on 256
IMG_SIZES = [8, 16, 32, 64, 128, 256, 512]


PATH = '../dataset_sample/'

# number of color channels we want to use
# set to 1 to convert to grayscale
# set to 3 to use color images
COLOR_CHANNELS = 3

## helper function to load and parse annotations (copied from course)


In [29]:
annotations = dict()

for condition in CONDITIONS:
    with open(f'{PATH}_annotations/{condition}.json') as f:
        annotations[condition] = json.load(f)

## helper function to pre-process images (color channel conversion and resizing) (also copied from course and adjusted slightly to take size as an arg)

In [30]:
def preprocess_image(img, size):
    if COLOR_CHANNELS == 1:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_resized = cv2.resize(img, (size, size)) # Use this tuple directly to make it loopable better later
    return img_resized

## Load images and annotations. Taken from the course and then adjusted because we of course need 5 different lists this time

In [31]:
# We can use dicts and store lists using the size as a key
images = {} # stores actual image data
for size in IMG_SIZES:
    images[size] = []
labels = {} # stores labels (as integer - because this is what our network needs)
for size in IMG_SIZES:
    labels[size] = []
label_names = {} # maps label ints to their actual categories so we can understand predictions later
for size in IMG_SIZES:
    label_names[size] = []

# loop over all conditions
# loop over all files in the condition's directory
# read the image and corresponding annotation
# crop image to the region of interest
# preprocess image
# store preprocessed image and label in corresponding lists
for condition in CONDITIONS:
    for filename in tqdm(os.listdir(f'{PATH}/{condition}')):
        # extract unique ID from file name
        UID = filename.split('.')[0]
        img = cv2.imread(f'{PATH}/{condition}/{filename}')

        # get annotation from the dict we loaded earlier
        try:
            annotation = annotations[condition][UID]
        except Exception as e:
            print(e)
            continue

        # iterate over all hands annotated in the image
        for i, bbox in enumerate(annotation['bboxes']):
            # annotated bounding boxes are in the range from 0 to 1
            # therefore we have to scale them to the image size
            x1 = int(bbox[0] * img.shape[1])
            y1 = int(bbox[1] * img.shape[0])
            w = int(bbox[2] * img.shape[1])
            h = int(bbox[3] * img.shape[0])
            x2 = x1 + w
            y2 = y1 + h

            # crop image to the bounding box and apply pre-processing
            crop = img[y1:y2, x1:x2]

            # We need to repeat this part for every size this time unlike in the course where we only did it once
            for size in IMG_SIZES:

                preprocessed = preprocess_image(crop, size)

                # get the annotated hand's label
                # if we have not seen this label yet, add it to the list of labels
                label = annotation['labels'][i]
                if label == "no_gesture":
                    continue
                if label not in label_names[size]:
                    label_names[size].append(label)

                label_index = label_names[size].index(label)

                images[size].append(preprocessed)
                labels[size].append(label_index)

100%|██████████| 250/250 [00:08<00:00, 29.48it/s]


## split data set into train and test

x is for the actual data, y is for the label (this is convention). We copy this from the course and run it for each image size

In [32]:
X_train = {}
X_test = {}
y_train = {}
y_test = {}
for size in IMG_SIZES:
    X_train_subset, X_test_subset, y_train_subset, y_test_subset = train_test_split(images[size], labels[size], test_size=0.2, random_state=42)
    X_train[size] = X_train_subset
    X_test[size] = X_test_subset
    y_train[size] = y_train_subset
    y_test[size] = y_test_subset


## transform data sets into a format compatible with our neural network

image data has to be a numpy array with following dimensions: [image_id, y_axis, x_axis, color_channels]

furthermore, scale all values to a range of 0 to 1

training data has to be converted to a categorial vector ("one hot"):

[3] --> [0, 0, 0, 1, 0, ..., 0]

This is also copied from the course script and then adjusted to run 5 times

In [33]:
train_label = {}
test_label = {}

for size in IMG_SIZES:
    X_train[size] = np.array(X_train[size]).astype('float32')
    X_train[size] /= 255

    X_test[size] = np.array(X_test[size]).astype('float32')
    X_test[size] /= 255

    y_train_one_hot = to_categorical(y_train[size])
    y_test_one_hot = to_categorical(y_test[size])

    train_label[size] = y_train_one_hot
    test_label[size] = y_test_one_hot

    X_train[size] = X_train[size].reshape(-1, size, size, COLOR_CHANNELS)
    X_test[size] = X_test[size].reshape(-1, size, size, COLOR_CHANNELS)



## Adding a check if everything went right

In [34]:
for size in IMG_SIZES:
    print(
        size,
        X_train[size].shape,
        X_test[size].shape,
        train_label[size].shape,
        test_label[size].shape
    )

8 (400, 8, 8, 3) (100, 8, 8, 3) (400, 2) (100, 2)
16 (400, 16, 16, 3) (100, 16, 16, 3) (400, 2) (100, 2)
32 (400, 32, 32, 3) (100, 32, 32, 3) (400, 2) (100, 2)
64 (400, 64, 64, 3) (100, 64, 64, 3) (400, 2) (100, 2)
128 (400, 128, 128, 3) (100, 128, 128, 3) (400, 2) (100, 2)
256 (400, 256, 256, 3) (100, 256, 256, 3) (400, 2) (100, 2)
512 (400, 512, 512, 3) (100, 512, 512, 3) (400, 2) (100, 2)


## Create a model

We will copy the model that was also used in the course

In [35]:
# variables for hyperparameters, we will keep these the same for each model
batch_size = 8
epochs = 50
num_classes = len(label_names[IMG_SIZES[0]]) # adjust this line
activation = 'relu'
activation_conv = 'relu'  # LeakyReLU
layer_count = 2
num_neurons = 64

models = {}
reducers = {}
stoppers = {}

for size in IMG_SIZES:

    # define model structure
    # with keras, we can use a model's add() function to add layers to the network one by one
    model = Sequential()

    # data augmentation (this can also be done beforehand - but don't augment the test dataset!)
    model.add(RandomFlip('horizontal'))
    model.add(RandomContrast(0.1))
    #model.add(RandomBrightness(0.1))
    #model.add(RandomRotation(0.2))

    # first, we add some convolution layers followed by max pooling
    model.add(Conv2D(64, kernel_size=(9, 9), activation=activation_conv, input_shape=(size, size, COLOR_CHANNELS), padding='same'))
    model.add(MaxPooling2D(pool_size=(4, 4), padding='same'))

    model.add(Conv2D(32, (5, 5), activation=activation_conv, padding='same'))
    model.add(MaxPooling2D(pool_size=(3, 3), padding='same'))

    model.add(Conv2D(32, (3, 3), activation=activation_conv, padding='same'))
    model.add(MaxPooling2D(pool_size=(2, 2), padding='same'))

    # dropout layers can drop part of the data during each epoch - this prevents overfitting
    model.add(Dropout(0.2))

    # after the convolution layers, we have to flatten the data so it can be fed into fully connected layers
    model.add(Flatten())

    # add some fully connected layers ("Dense")
    for i in range(layer_count - 1):
        model.add(Dense(num_neurons, activation=activation))

    model.add(Dense(num_neurons, activation=activation))

    # for classification, the last layer has to use the softmax activation function, which gives us probabilities for each category
    model.add(Dense(num_classes, activation='softmax'))

    # specify loss function, optimizer and evaluation metrics
    # for classification, categorial crossentropy is used as a loss function
    # use the adam optimizer unless you have a good reason not to
    model.compile(loss=categorical_crossentropy, optimizer="adam", metrics=['accuracy'])
    models[size] = model

    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=0.0001)
    stop_early = EarlyStopping(monitor='val_loss', patience=3)
    reducers[size] = reduce_lr
    stoppers[size] = stop_early

# define callback functions that react to the model's behavior during training
# in this example, we reduce the learning rate once we get stuck and early stopping
# to cancel the training if there are no improvements for a certain amount of epochs
# We can reuse these functions which is why we keep them outside the loop
# MAybe we cant? It is not working properly


## Training

We will now train our 5 models. This could take quite some time and it hopefully does not crash

In [37]:
import time

histories = {}
times = {}

for size in IMG_SIZES:
    start = time.time()
    print(f'Training model for images with size {size}x{size}')
    histories[size] = models[size].fit(
        X_train[size],
        train_label[size],
        batch_size=batch_size,
        epochs=epochs,
        verbose=1,
        validation_data=(X_test[size], test_label[size]),
        callbacks=[reducers[size], stoppers[size]]
    )
    end = time.time()
    times[size] = end - start
    print(f'Training for image size {size} took {times[size]} seconds')

Training model for images with size 8x8
Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9600 - loss: 0.1263 - val_accuracy: 0.9300 - val_loss: 0.2327 - learning_rate: 1.0000e-04
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9600 - loss: 0.1262 - val_accuracy: 0.9100 - val_loss: 0.2315 - learning_rate: 1.0000e-04
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9575 - loss: 0.1259 - val_accuracy: 0.9300 - val_loss: 0.2291 - learning_rate: 1.0000e-04
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9700 - loss: 0.1037 - val_accuracy: 0.9300 - val_loss: 0.2425 - learning_rate: 1.0000e-04
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9650 - loss: 0.1195 - val_accuracy: 0.9300 - val_loss: 0.2253 - learning_rate: 1.0000e-04
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9650 - loss: 0.1138 - val_accuracy: 0.9300 - val_loss: 0.2269 - learning_rate: 1.0000e-04
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━

## Check

Quick sanity check

In [38]:
# Quick check if val acc changed
for size in IMG_SIZES:
    print(
        size,
        max(histories[size].history['val_accuracy'])
    )

8 0.9300000071525574
16 0.9800000190734863
32 0.9700000286102295
64 0.9800000190734863
128 0.9700000286102295
256 0.9900000095367432
512 0.9399999976158142


# Time needed
Let us also check how long it took to train each model

In [39]:
for size in IMG_SIZES:
    print(size, times[size])

8 4.935735702514648
16 3.947930097579956
32 1.9839966297149658
64 4.570764541625977
128 8.839795351028442
256 27.722820520401
512 308.1340317726135


## Thought process

While the models are learning I want to share my thought process here

We know that doubling image size quadruples the datapoints we have (8 by 8 px -> 64px; 16 by 16 px -> 256px). Since each pixel represents one input node we get
a lot of nodes and way more input if we increse image size.

Therefore, I theorize that training will take longer for larger images.

Another thought is that more data offers more possibility for interpretation. This means that there are more possible patterns to be found but there is also more noise. I believe that if we have enough training the accuracy of models using larger images will surpass those with smaller images. However I also believe that without enough training models using larger images could perform worse due to the additional noise.

## Some results

We do see that training time decreases at first but then increases again. One logical explanation could be that images to small do not provide enough data to find patterns. We can also see that for 512 the time is way longer than for 256. This could mean that such a large amount of inputs comes with too much noise and therefore performs worse
